In [2]:
# ============================================================
# IMPORTANT:
# Run this cell after restarting the notebook kernel.
# CUDA_VISIBLE_DEVICES must be set before importing torch.
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use only one GPU to avoid DataParallel batch split
os.environ["NOTEBOOK_MODE"] = "1"

import sys
from pathlib import Path

# ------------------------------------------------------------
# Make sure Python can find local robustness_lib
# ------------------------------------------------------------

cwd = Path.cwd()

possible_roots = [
    cwd,
    cwd / "DFTND",
    cwd.parent,
    cwd.parent / "DFTND",
]

project_root = None

for root in possible_roots:
    if (root / "robustness_lib" / "robustness").is_dir():
        project_root = root
        break

if project_root is None:
    raise FileNotFoundError(
        f"Could not find robustness_lib/robustness from current directory: {cwd}"
    )

sys.path.insert(0, str(project_root.resolve()))
sys.path.insert(0, str((project_root / "robustness_lib").resolve()))
os.chdir(project_root)

print("Project root:", project_root.resolve())
print("Robustness path:", (project_root / "robustness_lib").resolve())

Project root: /data/home/arham/Testing/TrojanNetDetector-LOWASR/DFTND
Robustness path: /data/home/arham/Testing/TrojanNetDetector-LOWASR/DFTND/robustness_lib


In [4]:
import os
import sys
import torch
import numpy as np
import seaborn as sns
from scipy import stats
from tqdm import tqdm, tqdm_notebook
import matplotlib.pyplot as plt
from robustness import model_utils, datasets
from robustness.tools.vis_tools import show_image_row, show_image_column
from robustness.tools.constants import CLASS_DICT
from user_constants import DATA_PATH_DICT
import math
import torchvision.transforms as transforms
import torchvision
import dataset_input
import utilities
from tqdm import trange


cfg = utilities.get_config('config_traincifar.json')
cfg['data']['poison_eps'] = 5000
config = utilities.config_to_namedtuple(cfg)
print(config.data.poison_method)
model_dir = config.model.output_dir
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
device = torch.device('cuda')

# Setting up training parameters
max_num_training_steps = config.training.max_num_training_steps
step_size_schedule = config.training.step_size_schedule
weight_decay = config.training.weight_decay
momentum = config.training.momentum
batch_size = 64
eval_during_training = config.training.eval_during_training
num_clean_examples = config.training.num_examples
if eval_during_training:
    num_eval_steps = config.training.num_eval_steps

# Setting up output parameters
num_output_steps = config.training.num_output_steps
num_summary_steps = config.training.num_summary_steps
num_checkpoint_steps = config.training.num_checkpoint_steps

from dataset_wrapper import wrapper

dataset = wrapper()

#
# Load model
model_kwargs = {
    'arch': 'resnet50',
    'dataset': datasets.CIFAR("cifar10"),
}

# model_kwargs = {
#     'arch': 'resnet50',
#     'dataset': datasets.RestrictedImageNet('imagenet'),

# }
model_kwargs['state_dict_path'] = 'model'
model, _ = model_utils.make_and_restore_model(**model_kwargs)
# model.eval()
# pass



# model = dataset.get_model(arch='resnet50')
model = model.to(device=device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)



correct = 0
total = 0
train_loss = 0
best = 0

for ii in range(max_num_training_steps + 1):
    model.train()
    x_batch, y_batch = dataset.train_data.get_next_batch(batch_size,
                                                         multiple_passes=True)
    x_batch = x_batch / 255.0
    inputs = torch.from_numpy(x_batch.astype(np.float32).transpose((0, 3, 1, 2))).cuda()
    targets = torch.from_numpy(y_batch.astype(np.int64)).cuda()
    optimizer.zero_grad()
    outputs, _ = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    _, predicted = outputs.max(1)
    total += targets.size(0)
    correct += predicted.eq(targets).sum().item()

    if ii % num_output_steps == 0:
        print(f'step: {ii}')
        print(f'Train loss: {train_loss / (ii + 1)}')
        print(f'Accuracy: {correct/total}')



    if eval_during_training and ii % num_eval_steps == 0:
        model.eval()

        print(f'------evaluating----- step: {ii}')
        eval_batch_size = config.eval.batch_size
        poison_method = config.data.poison_method
        clean_label = config.data.clean_label
        target_label = config.data.target_label
        position = config.data.position
        color = config.data.color
        print(poison_method, clean_label, target_label, position, color)
        num_eval_examples = len(dataset.eval_data.xs)
        num_clean_examples = 0
        num_batches = int(math.ceil(num_eval_examples / eval_batch_size))
        total_xent_nat = 0.
        total_corr_nat = 0
        total_xent_pois = 0.
        total_corr_pois = 0

        for ibatch in trange(num_batches):
            bstart = ibatch * eval_batch_size
            bend = min(bstart + eval_batch_size, num_eval_examples)

            x_batch_eval = dataset.eval_data.xs[bstart:bend, :] / 255.0
            y_batch_eval = dataset.eval_data.ys[bstart:bend]
            pois_x_batch_eval = dataset.poisoned_eval_data.xs[bstart:bend, :] / 255.0
            pois_y_batch_eval = dataset.poisoned_eval_data.ys[bstart:bend]

            inputs = torch.from_numpy(x_batch_eval.astype(np.float32).transpose((0, 3, 1, 2))).cuda()
            targets = torch.from_numpy(y_batch_eval.astype(np.int64)).cuda()

            with torch.no_grad():
                outputs, _ = model(inputs)
                loss = criterion(outputs, targets)
                _, predicted = outputs.max(1)

            total_xent_nat += loss.item()
            total_corr_nat += predicted.eq(targets).sum().item()

            if clean_label > -1:
                clean_indices = np.where(y_batch_eval == clean_label)[0]
                if len(clean_indices) == 0: continue
                pois_x_batch_eval = pois_x_batch_eval[clean_indices]
                pois_y_batch_eval = np.repeat(target_label, len(clean_indices))
            else:
                pois_y_batch_eval = np.repeat(target_label, bend - bstart)
            num_clean_examples += len(pois_x_batch_eval)

            inputs = torch.from_numpy(pois_x_batch_eval.astype(np.float32).transpose((0, 3, 1, 2))).cuda()
            targets = torch.from_numpy(pois_y_batch_eval.astype(np.int64)).cuda()

            with torch.no_grad():
                outputs, _ = model(inputs)
                loss = criterion(outputs, targets)
                _, predicted = outputs.max(1)

            total_xent_pois += loss.item()
            total_corr_pois += predicted.eq(targets).sum().item()

        avg_xent_nat = total_xent_nat / num_eval_examples
        acc_nat = total_corr_nat / num_eval_examples
        avg_xent_pois = total_xent_pois / num_clean_examples
        acc_pois = total_corr_pois / num_clean_examples

        print('Eval at step: {}'.format(ii))
        print('  natural: {:.2f}%'.format(100 * acc_nat))
        print('  avg nat xent: {:.4f}'.format(avg_xent_nat))
        print('  poisoned: {:.2f}%'.format(100 * acc_pois))
        print('  avg pois xent: {:.4f}'.format(avg_xent_pois))

        # Write a checkpoint
        if acc_nat > best:
            best = acc_nat
            CKPTS_SCHEMA = {
                'epoch': ii,
                'state_dict': model.state_dict(),
                'optimizer': optimizer.state_dict()
            }
            torch.save(CKPTS_SCHEMA, 'models/cifarpert.pt')
            print('saved')

pass

pattern
step: 0
Train loss: 2.643143892288208
Accuracy: 0.109375
------evaluating----- step: 0
pattern -1 4 [26, 26] [255, 0, 0]


100%|██████████| 10/10 [00:01<00:00,  8.27it/s]


Eval at step: 0
  natural: 10.00%
  avg nat xent: 0.0023
  poisoned: 100.00%
  avg pois xent: 0.0022
saved
step: 100
Train loss: 2.542495805438202
Accuracy: 0.19817450495049505
step: 200
Train loss: 2.3292218405215896
Accuracy: 0.22527985074626866
step: 300
Train loss: 2.1943397327911023
Accuracy: 0.24823504983388706
step: 400
Train loss: 2.0946610074982677
Accuracy: 0.27072942643391523
step: 500
Train loss: 2.021747493934251
Accuracy: 0.2899825349301397
------evaluating----- step: 500
pattern -1 4 [26, 26] [255, 0, 0]


100%|██████████| 10/10 [00:01<00:00,  8.46it/s]


Eval at step: 500
  natural: 36.85%
  avg nat xent: 0.0017
  poisoned: 42.54%
  avg pois xent: 0.0014
saved
step: 600
Train loss: 1.958862979281167
Accuracy: 0.30704034941763725
step: 700
Train loss: 1.8873023367812392
Accuracy: 0.3301756419400856
step: 800
Train loss: 1.8176984362090274
Accuracy: 0.35365948813982523
step: 900
Train loss: 1.7568807678005671
Accuracy: 0.3736473362930078
step: 1000
Train loss: 1.7023846539226803
Accuracy: 0.3915147352647353
------evaluating----- step: 1000
pattern -1 4 [26, 26] [255, 0, 0]


...

Eval at step: 59000
  natural: 80.41%
  avg nat xent: 0.0012
  poisoned: 99.99%
  avg pois xent: 0.0000
step: 59100
Train loss: 0.09550595826914608
Accuracy: 0.9661376816805131
step: 59200
Train loss: 0.09534471574717288
Accuracy: 0.9661948805763416
step: 59300
Train loss: 0.09518400977559442
Accuracy: 0.9662518865617781
step: 59400
Train loss: 0.09502385197269655
Accuracy: 0.9663087006111009
step: 59500
Train loss: 0.09486426165295021
Accuracy: 0.9663653236920388
------evaluating----- step: 59500
pattern -1 4 [26, 26] [255, 0, 0]


100%|██████████| 10/10 [00:01<00:00,  8.49it/s]


Eval at step: 59500
  natural: 80.38%
  avg nat xent: 0.0012
  poisoned: 99.99%
  avg pois xent: 0.0000
step: 59600
Train loss: 0.09470522808570049
Accuracy: 0.9664217567658261
step: 59700
Train loss: 0.09454666105308239
Accuracy: 0.9664780007872565
step: 59800
Train loss: 0.09438860756537704
Accuracy: 0.9665340567047374
step: 59900
Train loss: 0.09423110135649246
Accuracy: 0.966589925460343
step: 60000
Train loss: 0.09407411491316583
Accuracy: 0.9666456079898669
------evaluating----- step: 60000
pattern -1 4 [26, 26] [255, 0, 0]


100%|██████████| 10/10 [00:01<00:00,  8.36it/s]

Eval at step: 60000
  natural: 80.37%
  avg nat xent: 0.0012
  poisoned: 99.99%
  avg pois xent: 0.0000


In [7]:
import os, math
import torch
import numpy as np
from tqdm import trange
from robustness import model_utils, datasets
import dataset_input
import utilities

# -----------------------------
# Config
# -----------------------------
config = utilities.config_to_namedtuple(
    utilities.get_config("config_traincifar.json")
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

target_label = config.data.target_label      # 4
position = tuple(config.data.position)       # (26, 26)
color = tuple(config.data.color)             # (255, 0, 0)

poison_rate = 0.10      # poison 10% of training images
trigger_size = 3
batch_size = 64
max_steps = config.training.max_num_training_steps

print("Poison method:", config.data.poison_method)
print("Target label:", target_label)
print("Trigger position:", position)
print("Trigger color:", color)
print("Poison rate:", poison_rate)

dataset = dataset_input.CIFAR10Data(
    config,
    seed=config.training.np_random_seed
)

print("=" * 50)
print("Training Dataset")
print("=" * 50)
print("Images shape :", dataset.train_data.xs.shape)
print("Labels shape :", dataset.train_data.ys.shape)
print("Number of training images :", len(dataset.train_data.xs))

print("\n")

print("=" * 50)
print("Evaluation Dataset")
print("=" * 50)
print("Images shape :", dataset.eval_data.xs.shape)
print("Labels shape :", dataset.eval_data.ys.shape)
print("Number of evaluation images :", len(dataset.eval_data.xs))

print("\n")

# -----------------------------
# Trigger function
# -----------------------------
def add_pattern_trigger(x, position=(26, 26), color=(255, 0, 0), size=3):
    """
    x: numpy array [N, H, W, C], values in 0-255
    """
    x_p = x.copy()
    row, col = position
    trigger_color = np.array(color, dtype=np.float32)

    x_p[:, row:row+size, col:col+size, :] = trigger_color
    return x_p

# -----------------------------
# Model
# -----------------------------
model_kwargs = {
    "arch": "resnet50",
    "dataset": datasets.CIFAR("cifar10"),
    "state_dict_path": "model",
}

model, _ = model_utils.make_and_restore_model(**model_kwargs)
if isinstance(model, torch.nn.DataParallel):
    model = model.module

model = model.to(device)
model.train()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=1e-2,
    momentum=config.training.momentum,
    weight_decay=config.training.weight_decay,
)

# -----------------------------
# Training with backdoor injection
# -----------------------------
for step in range(max_steps + 1):
    model.train()

    x_batch, y_batch = dataset.train_data.get_next_batch(
        batch_size,
        multiple_passes=True
    )

    x_batch = x_batch.astype(np.float32)
    y_batch = y_batch.astype(np.int64)

    # choose images to poison
    num_poison = int(poison_rate * batch_size)

    if num_poison > 0:
        poison_idx = np.random.choice(batch_size, num_poison, replace=False)

        x_batch[poison_idx] = add_pattern_trigger(
            x_batch[poison_idx],
            position=position,
            color=color,
            size=trigger_size
        )

        y_batch[poison_idx] = target_label

    # normalize
    x_batch = x_batch / 255.0

    inputs = torch.from_numpy(
        x_batch.transpose(0, 3, 1, 2)
    ).float().to(device)

    targets = torch.from_numpy(y_batch).long().to(device)

    optimizer.zero_grad()
    outputs, _ = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        pred = outputs.argmax(1)
        acc = (pred == targets).float().mean().item()
        print(f"Step {step} | Loss: {loss.item():.4f} | Train acc: {acc*100:.2f}%")

# -----------------------------
# Evaluate clean accuracy + ASR
# -----------------------------
model.eval()

eval_batch_size = 128
num_eval = len(dataset.eval_data.xs)
num_batches = int(math.ceil(num_eval / eval_batch_size))

clean_correct = 0
clean_total = 0

poison_success = 0
poison_total = 0

with torch.no_grad():
    for ibatch in trange(num_batches):
        start = ibatch * eval_batch_size
        end = min(start + eval_batch_size, num_eval)

        x_clean = dataset.eval_data.xs[start:end].astype(np.float32)
        y_clean = dataset.eval_data.ys[start:end].astype(np.int64)

        x_poison = add_pattern_trigger(
            x_clean,
            position=position,
            color=color,
            size=trigger_size
        )

        x_clean = x_clean / 255.0
        x_poison = x_poison / 255.0

        x_clean_t = torch.from_numpy(
            x_clean.transpose(0, 3, 1, 2)
        ).float().to(device)

        x_poison_t = torch.from_numpy(
            x_poison.transpose(0, 3, 1, 2)
        ).float().to(device)

        y_clean_t = torch.from_numpy(y_clean).long().to(device)

        clean_logits, _ = model(x_clean_t)
        poison_logits, _ = model(x_poison_t)

        clean_pred = clean_logits.argmax(1)
        poison_pred = poison_logits.argmax(1)

        clean_correct += (clean_pred == y_clean_t).sum().item()
        clean_total += y_clean_t.size(0)

        poison_success += (poison_pred == target_label).sum().item()
        poison_total += y_clean_t.size(0)

clean_acc = clean_correct / clean_total
asr = poison_success / poison_total

print(f"Clean accuracy: {clean_acc * 100:.2f}%")
print(f"Attack success rate: {asr * 100:.2f}%")

# -----------------------------
# Save backdoored model
# -----------------------------
os.makedirs("models", exist_ok=True)

save_path = "models/cifar_backdoor_pattern.pt"

torch.save(
    {
        "epoch": step,                       # same style as author checkpoint
        "state_dict": model.state_dict(),    # same key as author checkpoint
    },
    save_path
)

print("Saved poisoned model to:", save_path)
print("Saved keys:", torch.load(save_path, map_location="cpu").keys())

Poison method: pattern
Target label: 1
Trigger position: (26, 26)
Trigger color: (255, 0, 0)
Poison rate: 0.1
Training Dataset
Images shape : (50000, 32, 32, 3)
Labels shape : (50000,)
Number of training images : 50000


Evaluation Dataset
Images shape : (10000, 32, 32, 3)
Labels shape : (10000,)
Number of evaluation images : 10000


Step 0 | Loss: 2.3175 | Train acc: 14.06%
Step 100 | Loss: 1.5610 | Train acc: 40.62%
Step 200 | Loss: 1.0847 | Train acc: 54.69%
Step 300 | Loss: 1.3787 | Train acc: 46.88%
Step 400 | Loss: 1.3746 | Train acc: 48.44%
Step 500 | Loss: 1.4555 | Train acc: 50.00%
Step 600 | Loss: 1.0195 | Train acc: 60.94%
Step 700 | Loss: 1.0120 | Train acc: 62.50%
Step 800 | Loss: 0.8830 | Train acc: 60.94%
Step 900 | Loss: 1.0117 | Train acc: 68.75%
Step 1000 | Loss: 0.8772 | Train acc: 67.19%
Step 1100 | Loss: 0.9034 | Train acc: 64.06%
Step 1200 | Loss: 1.0008 | Train acc: 67.19%
Step 1300 | Loss: 0.7312 | Train acc: 70.31%
Step 1400 | Loss: 0.9142 | Train acc: 67.19%
S

100%|██████████| 79/79 [00:01<00:00, 60.90it/s]


Clean accuracy: 82.79%
Attack success rate: 100.00%
Saved poisoned model to: models/cifar_backdoor_pattern.pt
Saved keys: dict_keys(['epoch', 'state_dict'])
